In [1]:
import pandas as pd
import numpy as np
import polars as pl
import sklearn
import xgboost as xgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("polars:", pl.__version__)
print("scikit-learn:", sklearn.__version__)
print("xgboost:", xgb.__version__)
print("shap:", shap.__version__)

print("\nAmbiente configurado com sucesso!")

pandas: 3.0.3
numpy: 2.4.6
polars: 1.42.0
scikit-learn: 1.9.0
xgboost: 3.3.0
shap: 0.52.0

Ambiente configurado com sucesso!


In [2]:
df = pd.read_csv('../data/raw/application_train.csv')

print("Shape:", df.shape)
print("\nTaxa de default (TARGET):")
print(df['TARGET'].value_counts(normalize=True))

Shape: (307511, 122)

Taxa de default (TARGET):
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64


## Dataset Overview

The dataset contains 307,511 loan applications with 122 features.
The default rate (TARGET=1) is 8%, reflecting Home Credit's profile:
a consumer finance provider targeting borrowers with little or no formal
credit history, combining higher-risk borrower profiles with real estate collateral.

The class imbalance (92% vs 8%) makes accuracy an inadequate metric.
Primary evaluation metrics: AUC-ROC, KS statistic, Gini coefficient.

Case Description:

Many people struggle to get loans due to insufficient or non-existent credit histories. And, unfortunately, this population is often taken advantage of by untrustworthy lenders.

Home Credit Group

Home Credit strives to broaden financial inclusion for the unbanked population by providing a positive and safe borrowing experience. In order to make sure this underserved population has a positive loan experience, Home Credit makes use of a variety of alternative data--including telco and transactional information--to predict their clients' repayment abilities.

While Home Credit is currently using various statistical and machine learning methods to make these predictions, they're challenging Kagglers to help them unlock the full potential of their data. Doing so will ensure that clients capable of repayment are not rejected and that loans are given with a principal, maturity, and repayment calendar that will empower their clients to be successful.

In [4]:
# Variable types and missing values overview
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

summary = pd.DataFrame({
    'dtype': df.dtypes,
    'missing_count': missing,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

print("=== Variable Types ===")
print(df.dtypes.value_counts())

print("\n=== Top 20 variables with most missing values ===")
print(summary[summary['missing_count'] > 0].head(20))

print(f"\nTotal variables with missing values: {(missing > 0).sum()}")
print(f"Total variables with >50% missing: {(missing_pct > 50).sum()}")

=== Variable Types ===
float64    65
int64      41
str        16
Name: count, dtype: int64

=== Top 20 variables with most missing values ===
                            dtype  missing_count  missing_pct
COMMONAREA_AVG            float64         214865        69.87
COMMONAREA_MODE           float64         214865        69.87
COMMONAREA_MEDI           float64         214865        69.87
NONLIVINGAPARTMENTS_MEDI  float64         213514        69.43
NONLIVINGAPARTMENTS_MODE  float64         213514        69.43
NONLIVINGAPARTMENTS_AVG   float64         213514        69.43
FONDKAPREMONT_MODE            str         210295        68.39
LIVINGAPARTMENTS_AVG      float64         210199        68.35
LIVINGAPARTMENTS_MEDI     float64         210199        68.35
LIVINGAPARTMENTS_MODE     float64         210199        68.35
FLOORSMIN_MODE            float64         208642        67.85
FLOORSMIN_AVG             float64         208642        67.85
FLOORSMIN_MEDI            float64         208642    

In [ ]:
# Separate numeric and categorical variables
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['str']).columns.tolist()

# Remove TARGET from numeric list
numeric_cols = [c for c in numeric_cols if c != 'TARGET']

print(f"Numeric variables: {len(numeric_cols)}")
print(f"Categorical variables: {len(categorical_cols)}")

print("\n=== Categorical variables and unique values ===")
for col in categorical_cols:
    n_unique = df[col].nunique()
    top_values = df[col].value_counts().head(3).to_dict()
    print(f"\n{col} ({n_unique} unique values)")
    print(f"  Top 3: {top_values}")

Numeric variables: 105
Categorical variables: 16

=== Categorical variables and unique values ===

NAME_CONTRACT_TYPE (2 unique values)
  Top 3: {'Cash loans': 278232, 'Revolving loans': 29279}

CODE_GENDER (3 unique values)
  Top 3: {'F': 202448, 'M': 105059, 'XNA': 4}

FLAG_OWN_CAR (2 unique values)
  Top 3: {'N': 202924, 'Y': 104587}

FLAG_OWN_REALTY (2 unique values)
  Top 3: {'Y': 213312, 'N': 94199}

NAME_TYPE_SUITE (7 unique values)
  Top 3: {'Unaccompanied': 248526, 'Family': 40149, 'Spouse, partner': 11370}

NAME_INCOME_TYPE (8 unique values)
  Top 3: {'Working': 158774, 'Commercial associate': 71617, 'Pensioner': 55362}

NAME_EDUCATION_TYPE (5 unique values)
  Top 3: {'Secondary / secondary special': 218391, 'Higher education': 74863, 'Incomplete higher': 10277}

NAME_FAMILY_STATUS (6 unique values)
  Top 3: {'Married': 196432, 'Single / not married': 45444, 'Civil marriage': 29775}

NAME_HOUSING_TYPE (6 unique values)
  Top 3: {'House / apartment': 272868, 'With parents': 14

In [8]:
desc = pd.read_csv('../data/raw/HomeCredit_columns_description.csv', 
                   encoding='latin-1')
print(desc.shape)
print(desc.head())
print(desc.columns.tolist())

(219, 5)
   Unnamed: 0                         Table                 Row  \
0           1  application_{train|test}.csv          SK_ID_CURR   
1           2  application_{train|test}.csv              TARGET   
2           5  application_{train|test}.csv  NAME_CONTRACT_TYPE   
3           6  application_{train|test}.csv         CODE_GENDER   
4           7  application_{train|test}.csv        FLAG_OWN_CAR   

                                         Description Special  
0                           ID of loan in our sample     NaN  
1  Target variable (1 - client with payment diffi...     NaN  
2        Identification if loan is cash or revolving     NaN  
3                               Gender of the client     NaN  
4                      Flag if the client owns a car     NaN  
['Unnamed: 0', 'Table', 'Row', 'Description', 'Special']


In [9]:
# Filter descriptions for main application table only
app_desc = desc[desc['Table'] == 'application_{train|test}.csv'].copy()
app_desc = app_desc[['Row', 'Description', 'Special']].reset_index(drop=True)

print(f"Variables described: {len(app_desc)}")
print("\n=== Full variable dictionary ===")
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 200)
print(app_desc)

Variables described: 122

=== Full variable dictionary ===
                              Row  \
0                      SK_ID_CURR   
1                          TARGET   
2              NAME_CONTRACT_TYPE   
3                     CODE_GENDER   
4                    FLAG_OWN_CAR   
5                 FLAG_OWN_REALTY   
6                    CNT_CHILDREN   
7                AMT_INCOME_TOTAL   
8                      AMT_CREDIT   
9                     AMT_ANNUITY   
10                AMT_GOODS_PRICE   
11                NAME_TYPE_SUITE   
12               NAME_INCOME_TYPE   
13            NAME_EDUCATION_TYPE   
14             NAME_FAMILY_STATUS   
15              NAME_HOUSING_TYPE   
16     REGION_POPULATION_RELATIVE   
17                     DAYS_BIRTH   
18                  DAYS_EMPLOYED   
19              DAYS_REGISTRATION   
20                DAYS_ID_PUBLISH   
21                    OWN_CAR_AGE   
22                     FLAG_MOBIL   
23                 FLAG_EMP_PHONE   
24              

In [12]:
# Identify numeric variables with at least one negative value
neg_vars = df[numeric_cols].lt(0).any()
neg_vars = neg_vars[neg_vars].index.tolist()

summary_neg = df[neg_vars].agg(['min', 'max', 'mean']).T
summary_neg['neg_count'] = df[neg_vars].lt(0).sum()

print(f"Total de variáveis numéricas com valores negativos: {len(neg_vars)}")
print(summary_neg.round(2))

Total de variáveis numéricas com valores negativos: 5
                            min       max      mean  neg_count
DAYS_BIRTH             -25229.0   -7489.0 -16037.00     307511
DAYS_EMPLOYED          -17912.0  365243.0  63815.05     252135
DAYS_REGISTRATION      -24672.0       0.0  -4986.12     307431
DAYS_ID_PUBLISH         -7197.0       0.0  -2994.20     307495
DAYS_LAST_PHONE_CHANGE  -4292.0       0.0   -962.86     269838


In [13]:
df.loc[df['DAYS_EMPLOYED'] == 365243, 'NAME_INCOME_TYPE'].value_counts()

NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22
Name: count, dtype: int64

In [ ]:
# in variable 'DAYS_EMPLOYED' we observe 252,135 negative values over 307,511 = 55,376
# comparing to 'NAME_INCOME_TYPE' we find 55,374. Will not further investigate the other two cases, because is a too small difference.

In [15]:
# DAYS_EMPLOYED has a sentinel value (365243) representing "not applicable",
# used mainly for Pensioner and Unemployed clients who have no current job start date.
# Strategy: flag first, then convert sentinel to NaN. XGBoost handles NaN natively;
# imputation strategy for the logistic regression scorecard will be revisited during
# feature engineering (likely WoE binning, which treats missing as its own bin).

df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype('int8')
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

print(df['DAYS_EMPLOYED_ANOM'].value_counts())
print(df['DAYS_EMPLOYED'].isna().sum())

DAYS_EMPLOYED_ANOM
0    252137
1     55374
Name: count, dtype: int64
55374


C:\Users\vitor\AppData\Local\Temp\ipykernel_65500\2364814432.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['DAYS_EMPLOYED_ANOM'] = (df['DAYS_EMPLOYED'] == 365243).astype('int8')


In [16]:
df.shape

(307511, 123)

In [ ]:
# just created our first new variable on the cell above

In [17]:
# Convert DAYS_* variables to positive, human-readable units, keeping the
# original Kaggle columns untouched for traceability. This is purely an
# interpretability transformation: it does not affect model performance,
# since tree-based splits are invariant to sign, and coefficient sign in
# logistic regression simply flips accordingly.

df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).round(1)
df['YEARS_REGISTRATION'] = (-df['DAYS_REGISTRATION'] / 365).round(1)
df['YEARS_ID_PUBLISH'] = (-df['DAYS_ID_PUBLISH'] / 365).round(1)
df['YEARS_LAST_PHONE_CHANGE'] = (-df['DAYS_LAST_PHONE_CHANGE'] / 365).round(1)

print(df[['DAYS_BIRTH', 'AGE_YEARS',
          'DAYS_REGISTRATION', 'YEARS_REGISTRATION',
          'DAYS_ID_PUBLISH', 'YEARS_ID_PUBLISH',
          'DAYS_LAST_PHONE_CHANGE', 'YEARS_LAST_PHONE_CHANGE']].head())

   DAYS_BIRTH  AGE_YEARS  DAYS_REGISTRATION  YEARS_REGISTRATION  \
0       -9461       25.9            -3648.0                10.0   
1      -16765       45.9            -1186.0                 3.2   
2      -19046       52.2            -4260.0                11.7   
3      -19005       52.1            -9833.0                26.9   
4      -19932       54.6            -4311.0                11.8   

   DAYS_ID_PUBLISH  YEARS_ID_PUBLISH  DAYS_LAST_PHONE_CHANGE  \
0            -2120               5.8                 -1134.0   
1             -291               0.8                  -828.0   
2            -2531               6.9                  -815.0   
3            -2437               6.7                  -617.0   
4            -3458               9.5                 -1106.0   

   YEARS_LAST_PHONE_CHANGE  
0                      3.1  
1                      2.3  
2                      2.2  
3                      1.7  
4                      3.0  


C:\Users\vitor\AppData\Local\Temp\ipykernel_65500\3878153282.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['AGE_YEARS'] = (-df['DAYS_BIRTH'] / 365).round(1)
C:\Users\vitor\AppData\Local\Temp\ipykernel_65500\3878153282.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['YEARS_REGISTRATION'] = (-df['DAYS_REGISTRATION'] / 365).round(1)
C:\Users\vitor\AppData\Local\Temp\ipykernel_65500\3878153282.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, w

In [18]:
# Same readability transformation applied to the remaining DAYS_* variable.
# NaN values (former 365243 sentinel) propagate correctly through the division.
df['YEARS_EMPLOYED'] = (-df['DAYS_EMPLOYED'] / 365).round(1)

C:\Users\vitor\AppData\Local\Temp\ipykernel_65500\3007858180.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['YEARS_EMPLOYED'] = (-df['DAYS_EMPLOYED'] / 365).round(1)


In [19]:
# Correlation of numeric variables with TARGET
correlations = df[numeric_cols + ['TARGET']].corr()['TARGET'].drop('TARGET')
correlations = correlations.abs().sort_values(ascending=False)

print("=== Top 20 variables by absolute correlation with TARGET ===")
print(correlations.head(20).round(4))

print("\n=== Bottom 10 (lowest correlation) ===")
print(correlations.tail(10).round(4))

=== Top 20 variables by absolute correlation with TARGET ===
EXT_SOURCE_3                   0.1789
EXT_SOURCE_2                   0.1605
EXT_SOURCE_1                   0.1553
DAYS_BIRTH                     0.0782
DAYS_EMPLOYED                  0.0750
REGION_RATING_CLIENT_W_CITY    0.0609
REGION_RATING_CLIENT           0.0589
DAYS_LAST_PHONE_CHANGE         0.0552
DAYS_ID_PUBLISH                0.0515
REG_CITY_NOT_WORK_CITY         0.0510
FLAG_EMP_PHONE                 0.0460
REG_CITY_NOT_LIVE_CITY         0.0444
FLAG_DOCUMENT_3                0.0443
FLOORSMAX_AVG                  0.0440
FLOORSMAX_MEDI                 0.0438
FLOORSMAX_MODE                 0.0432
DAYS_REGISTRATION              0.0420
AMT_GOODS_PRICE                0.0396
OWN_CAR_AGE                    0.0376
REGION_POPULATION_RELATIVE     0.0372
Name: TARGET, dtype: float64

=== Bottom 10 (lowest correlation) ===
FLAG_DOCUMENT_7               0.0015
FLAG_DOCUMENT_10              0.0014
FLAG_DOCUMENT_19              0.0014

In [20]:
pearson_corr = df[numeric_cols + ['TARGET']].corr(method='pearson')['TARGET'].drop('TARGET')
spearman_corr = df[numeric_cols + ['TARGET']].corr(method='spearman')['TARGET'].drop('TARGET')

comparison = pd.DataFrame({
    'pearson': pearson_corr,
    'spearman': spearman_corr,
    'gap': (spearman_corr.abs() - pearson_corr.abs())
}).sort_values('gap', ascending=False)

print("=== Variáveis onde Spearman > Pearson (indício de não-linearidade) ===")
print(comparison.head(15).round(4))

=== Variáveis onde Spearman > Pearson (indício de não-linearidade) ===
                              pearson  spearman     gap
YEARS_BEGINEXPLUATATION_MODE  -0.0090   -0.0271  0.0181
YEARS_BEGINEXPLUATATION_AVG   -0.0097   -0.0274  0.0177
YEARS_BEGINEXPLUATATION_MEDI  -0.0100   -0.0275  0.0176
OWN_CAR_AGE                    0.0376    0.0529  0.0153
AMT_INCOME_TOTAL              -0.0040   -0.0181  0.0141
LIVINGAREA_AVG                -0.0330   -0.0416  0.0086
LIVINGAREA_MEDI               -0.0327   -0.0411  0.0084
LIVINGAREA_MODE               -0.0307   -0.0391  0.0084
TOTALAREA_MODE                -0.0326   -0.0401  0.0075
AMT_REQ_CREDIT_BUREAU_QRT     -0.0020   -0.0085  0.0064
LIVINGAPARTMENTS_MEDI         -0.0246   -0.0308  0.0062
LIVINGAPARTMENTS_AVG          -0.0250   -0.0312  0.0061
LIVINGAPARTMENTS_MODE         -0.0234   -0.0292  0.0058
APARTMENTS_MODE               -0.0273   -0.0330  0.0057
APARTMENTS_AVG                -0.0295   -0.0350  0.0055


In [21]:
# reset numeric cols because we we created new numeric variables
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != 'TARGET']

In [22]:
# Correlation of numeric variables with TARGET
correlations = df[numeric_cols + ['TARGET']].corr()['TARGET'].drop('TARGET')
correlations = correlations.abs().sort_values(ascending=False)

print("=== Top 20 variables by absolute correlation with TARGET ===")
print(correlations.head(20).round(4))

print("\n=== Bottom 10 (lowest correlation) ===")
print(correlations.tail(10).round(4))

=== Top 20 variables by absolute correlation with TARGET ===
EXT_SOURCE_3                   0.1789
EXT_SOURCE_2                   0.1605
EXT_SOURCE_1                   0.1553
DAYS_BIRTH                     0.0782
AGE_YEARS                      0.0782
YEARS_EMPLOYED                 0.0750
DAYS_EMPLOYED                  0.0750
REGION_RATING_CLIENT_W_CITY    0.0609
REGION_RATING_CLIENT           0.0589
DAYS_LAST_PHONE_CHANGE         0.0552
YEARS_LAST_PHONE_CHANGE        0.0552
YEARS_ID_PUBLISH               0.0515
DAYS_ID_PUBLISH                0.0515
REG_CITY_NOT_WORK_CITY         0.0510
DAYS_EMPLOYED_ANOM             0.0460
FLAG_EMP_PHONE                 0.0460
REG_CITY_NOT_LIVE_CITY         0.0444
FLAG_DOCUMENT_3                0.0443
FLOORSMAX_AVG                  0.0440
FLOORSMAX_MEDI                 0.0438
Name: TARGET, dtype: float64

=== Bottom 10 (lowest correlation) ===
FLAG_DOCUMENT_7               0.0015
FLAG_DOCUMENT_10              0.0014
FLAG_DOCUMENT_19              0.0014

In [23]:
pearson_corr = df[numeric_cols + ['TARGET']].corr(method='pearson')['TARGET'].drop('TARGET')
spearman_corr = df[numeric_cols + ['TARGET']].corr(method='spearman')['TARGET'].drop('TARGET')

comparison = pd.DataFrame({
    'pearson': pearson_corr,
    'spearman': spearman_corr,
    'gap': (spearman_corr.abs() - pearson_corr.abs())
}).sort_values('gap', ascending=False)

print("=== Variáveis onde Spearman > Pearson (indício de não-linearidade) ===")
print(comparison.head(15).round(4))

=== Variáveis onde Spearman > Pearson (indício de não-linearidade) ===
                              pearson  spearman     gap
YEARS_BEGINEXPLUATATION_MODE  -0.0090   -0.0271  0.0181
YEARS_BEGINEXPLUATATION_AVG   -0.0097   -0.0274  0.0177
YEARS_BEGINEXPLUATATION_MEDI  -0.0100   -0.0275  0.0176
OWN_CAR_AGE                    0.0376    0.0529  0.0153
AMT_INCOME_TOTAL              -0.0040   -0.0181  0.0141
LIVINGAREA_AVG                -0.0330   -0.0416  0.0086
LIVINGAREA_MEDI               -0.0327   -0.0411  0.0084
LIVINGAREA_MODE               -0.0307   -0.0391  0.0084
TOTALAREA_MODE                -0.0326   -0.0401  0.0075
AMT_REQ_CREDIT_BUREAU_QRT     -0.0020   -0.0085  0.0064
LIVINGAPARTMENTS_MEDI         -0.0246   -0.0308  0.0062
LIVINGAPARTMENTS_AVG          -0.0250   -0.0312  0.0061
LIVINGAPARTMENTS_MODE         -0.0234   -0.0292  0.0058
APARTMENTS_MODE               -0.0273   -0.0330  0.0057
APARTMENTS_AVG                -0.0295   -0.0350  0.0055


In [24]:
df.shape

(307511, 128)

In [25]:
# Defragment the DataFrame after multiple individual column insertions
# during the negative-values treatment step. This resolves the
# PerformanceWarning without changing any data.
df = df.copy()